# 01. 탐색 및 실험 노트

실제 로직은 모두 `src/mission10/` 안의 모듈에 구현한다. 이 노트북은 그 함수들을 불러와
중간 결과를 눈으로 확인하고, `configs/`의 파라미터를 바꿔가며 실험한 결과를 비교하는 용도로만 쓴다.

In [1]:
import sys
sys.path.insert(0, "../src")

from mission10.config import load_config
from mission10 import preprocessing, embeddings, dataset, model, train, metrics, compare, visualize

config = load_config("../configs/base.yaml")
config

Config(data=DataConfig(source='20newsgroups', raw_path=None, text_column=None, label_column=None, test_ratio=0.2, val_ratio=0.1, seed=42), preprocessing=PreprocessingConfig(lowercase=True, remove_special_chars=True, remove_stopwords=True, min_token_len=1, max_len=280), embedding=EmbeddingConfig(method='word2vec', embedding_dim=100, window=5, min_count=1, freeze=False, glove_path=None), model=ModelConfig(type='lstm', hidden_size=128, num_layers=2, bidirectional=True, dropout=0.5), train=TrainConfig(batch_size=64, lr=0.005, epochs=10, seed=42), extra={})

In [2]:
df = preprocessing.load_raw_data(config.data)
df.shape

(18846, 2)

In [3]:
all_tokens = [preprocessing.tokenize(preprocessing.clean_text(t, config.preprocessing)) for t in df["text"][:100]]
vocab = preprocessing.build_vocab(all_tokens, min_count=config.embedding.min_count)
print(len(vocab))
print(list(vocab.items())[:10])

3876
[('<pad>', 0), ('<unk>', 1), ('sure', 2), ('bashers', 3), ('pens', 4), ('fans', 5), ('pretty', 6), ('confused', 7), ('lack', 8), ('kind', 9)]


In [4]:
train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = preprocessing.train_test_split_texts(
    df["text"].tolist(), df["label"].tolist(),
    config.data.test_ratio, config.data.val_ratio, config.data.seed
)

cleaned = [preprocessing.clean_text(t, config.preprocessing) for t in train_texts[:200]]
token_lists = [preprocessing.tokenize(t) for t in cleaned]
vocab = preprocessing.build_vocab(token_lists, min_count=config.embedding.min_count)

print(len(train_texts), len(val_texts), len(test_texts))
print(token_lists[0][:10])
print("vocab size:", len(vocab))

13568 1508 3770
['stereo', 'compressorlimiter', 'audio', 'logic', 'model', 'mt', 'gates', 'work', 'compressor', 'seems']
vocab size: 7445
